# 835 File Ingestion

The library is called edi_835_parser, but I did make some tweaks to a few of the files. I uploaded the folder of the module to the Sharepoint. You can tell by the modified dates which files I modified - I believe they are:

###### transaction_set/transaction_set
###### segments/organization


1) Download new 835 files from Blob to same directory as this script: hcmclientfilesv1/Riverview/Inbound/835/
2) Run the steps below - validate that the count returned from the "Count Files" cell matches the number of 835 files downloaded from Blob. You may have to delete previous day's output CSV from the directory if you did not already do that, as it will get counted and tried to process like any other file in the directory

In [1]:
from edi_835_parser import parse
import sys
import shutil
import os
import pandas as pd
import warnings

In [2]:
cwd = os.getcwd()
print(cwd)

c:\Users\AshwinJayakumar\Downloads\Cinq\HCM\835


In [3]:
class HiddenPrints:
    def __enter__(self):
        self._original_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')

    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout.close()
        sys.stdout = self._original_stdout

### Count Files

In [4]:
count = 0
for filename in os.listdir(cwd):

    if filename not in ('835_parsing.ipynb','failed','processed','edi_835_parser'):
        count+=1

print(count)

568


### Main File Parsing Step

In [5]:
warnings.filterwarnings('ignore')

overall_data=pd.DataFrame()
total_rows = 0
total_files_parsed = 0
successful_filenames = []
failed_files = 0
failed_filenames = []


for filename in os.listdir(cwd):

    if filename not in ('835_parsing.ipynb','failed','processed','edi_835_parser'):

        try:
            transaction_set = parse(filename)
            data = transaction_set.to_dataframe()
            data['filename'] = filename

    
            overall_data = pd.concat([overall_data, data], ignore_index=True)
    
            total_files_parsed+=1
            total_rows+=data.shape[0]
            successful_filenames.append(filename)
            os.rename(cwd + "\\" + filename, cwd + "\\processed\\" + filename)

        except:
            print(filename, "Failed !")
            failed_files+=1
            failed_filenames.append(filename)
            os.rename(cwd + "\\" + filename, cwd + "\\failed\\" + filename)

In [7]:
print('Success : ', total_files_parsed)
print('Failures : ', failed_files)
print(overall_data.shape)

Success :  568
Failures :  0
(70666, 44)


In [8]:
len(overall_data.filename.unique())

563

### Assuming no failed_files above, change the below to whatever name you want. I use today's date + underscore + number of original files

In [9]:
overall_data.to_csv('20250110_568.csv', index=False)

## Remaining Steps
3) Upload the above CSV to Blob: riverview/835
4) Switch ADF to the Atera-1476 branch (not been merged to main yet)
5) Run the pipeline in ADF "835_Output_To_DB" - this will load the output CSV to the database table Test.RVH_835
6) Run the pipeline in ADF "835_Manual_Move" - this will archive the existing Blob files from hcmclientfilesv1/Riverview/Inbound/835/ to hcmclientfilesv1/Riverview/Inbound/835/processed
7) Delete the output CSV from your directory, and any 835 files in the /processed folder on your local machine (they contain PHI)